In [1]:
# %pip install llama-index-packs-code-hierarchy

In [3]:
from llama_cpp.llama import Llama, LlamaGrammar
import httpx
from llama_index.core.node_parser import SentenceSplitter

from llama_cpp.llama import LlamaGrammar
import numpy as np
import pandas as pd
import torch
# from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.llms.llama_cpp import LlamaCPP
from llama_index.llms.llama_cpp.llama_utils import (
    messages_to_prompt,
    completion_to_prompt,
)
from llama_index.core import Settings
from llama_index.core import SimpleDirectoryReader, StorageContext
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.postgres import PGVectorStore
import textwrap
from llama_index.core import Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from transformers import AutoTokenizer
from llama_index.core import set_global_tokenizer

from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.query_engine import RouterQueryEngine

ImportError: cannot import name 'nn' from partially initialized module 'torch' (most likely due to a circular import) (/llm/.venv/lib/python3.11/site-packages/torch/__init__.py)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-14B-Instruct")
# tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct")
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

set_global_tokenizer(
    tokenizer.encode
)

In [ ]:
# r = httpx.get("https://ggplot2.tidyverse.org/reference/index.html")
# t = r.text
# soup = BeautifulSoup(t)

In [ ]:
# links = [a['href'] for a in soup.find_all('a', href=True)]

In [ ]:
# ggplot_links = [f"https://ggplot2.tidyverse.org/{i[3:]}" for i in links if i.startswith('../')]

In [ ]:
# urls = ggplot_links
# tags = []
# html_docs = []
# for i in urls:
#     r = httpx.get(i)
#     html_text = r.text
#     soup = BeautifulSoup(html_text)
#     tags.extend([tag.name for tag in soup.find_all()])
#     html_docs.append(Document(text=html_text))
# tags = list(set(tags))


In [ ]:
data_dir = "/notebooks/llm/crow_rag/ttlg/output/"
html_docs = []
for ext in ['*.html']:
    for path in Path(data_dir).rglob(ext):
        with open(path, 'r') as file:
            html_text = file.read().decode('windows-1252')
            # soup = BeautifulSoup(html_text)
            # tags.extend([tag.name for tag in soup.find_all()])
            html_docs.append(Document(text=file.read()))



    


In [ ]:
# The default tags are: ["p", "h1", "h2", "h3", "h4", "h5", "h6", "li", "b", "i", "u", "section"]
parser = HTMLNodeParser(tags=tags)
nodes = parser.get_nodes_from_documents(html_docs)
print(len(nodes))

In [ ]:
Settings.text_splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)

# per-index
index = VectorStoreIndex.from_documents(
    docs, storage_context=storage_context,
    transformations=[SentenceSplitter(chunk_size=512, chunk_overlap=50)], 
    show_progress=True
)

In [ ]:
# tags = []
# html_docs = []
# for i in f:
#     with open("/notebooks/llm/crow_rag/ttlg/output/"+i, 'rb') as file:
#         html_text = file.read().decode('windows-1252')
#         soup = BeautifulSoup(html_text)
#         tags.extend([tag.name for tag in soup.find_all()])
#         html_docs.append(Document(text=html_text))
# tags = list(set(tags))

In [ ]:
# r = httpx.get("https://monome.org/docs/crow/reference/")
# html = r.text
# soup = BeautifulSoup(r.text)
# tags = [tag.name for tag in soup.find_all()]

In [ ]:
import psycopg
def drop(name):
    with psycopg.connect("host=postgres dbname=grover user=grover password=grover") as conn:
        with conn.cursor() as cur:
            cur.execute(f"""
                drop table if exists {name};
                """)
            conn.commit()

drop("data_html")

In [ ]:
vector_store = PGVectorStore.from_params(
    database='grover',
    host='postgres',
    password='grover',
    port=5432,
    user='grover',
    table_name="html",
    embed_dim=384,  
    hnsw_kwargs={
        "hnsw_m": 14,
        "hnsw_ef_construction": 72,
        "hnsw_ef_search": 52,
        "hnsw_dist_method": "vector_cosine_ops",
    },
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [ ]:
# index = VectorStoreIndex(nodes, storage_context=storage_context, show_progress=True)

In [ ]:
# llm = HuggingFaceLLM(
#     context_window=8192,
#     max_new_tokens=2048,
#     generate_kwargs={"do_sample": True, 
#                      "eos_token_id": tokenizer.eos_token_id,
#                      "top_k": 7,
#                      "top_p": 0.3,
#                      "temperature": 0.05
#                     },
#     # query_wrapper_prompt=query_wrapper_prompt,
#     tokenizer_name="meta-llama/Llama-3.1-8B-Instruct",
#     model_name="meta-llama/Llama-3.1-8B-Instruct",
#     device_map="auto",
#     tokenizer_kwargs={"max_length": 2048},
#     model_kwargs={"torch_dtype": torch.float16},
# )
# Settings.llm = llm


In [ ]:
# def messages_to_prompt(messages):
#     prompt = ""
#     for message in messages:
#         if message.role == 'system':
#             prompt += f"<|system|>\n{message.content}</s>\n"
#         elif message.role == 'user':
#             prompt += f"<|user|>\n{message.content}</s>\n"
#         elif message.role == 'assistant':
#             prompt += f"<|assistant|>\n{message.content}</s>\n"

#     # ensure we start with a system prompt, insert blank if needed
#     if not prompt.startswith("<|system|>\n"):
#         prompt = "<|system|>\n</s>\n" + prompt

#     # add final assistant prompt
#     prompt = prompt + "<|assistant|>\n"

#     return prompt

# def completion_to_prompt(completion):
#     return f"<|system|>\n</s>\n<|user|>\n{completion}</s>\n<|assistant|>\n"

In [ ]:
# llm = LlamaCPP(
#     model_path="/hf_cache/models--bartowski--DeepSeek-Coder-V2-Lite-Instruct-GGUF/snapshots/8f248fa2072348f77a8bc37754e470de1f61866e/DeepSeek-Coder-V2-Lite-Instruct-Q6_K.gguf",
#     temperature=0,
#     max_new_tokens=2048,
#     context_window=16384,
#     generate_kwargs={
#         "repeat_penalty": 1.08,
#         "top_k": 0,
#         "top_p": 0},
#     model_kwargs={
#         "n_gpu_layers": -1,
#         # "grammar": grammar
#                  },
#     messages_to_prompt=messages_to_prompt,
#     completion_to_prompt=completion_to_prompt,
#     verbose=True,
# )

# Settings.llm = llm

In [ ]:
# llm = LlamaCPP(
#     model_path = "/hf_cache/models--Orenguteng--Llama-3.1-8B-Lexi-Uncensored-V2-GGUF/snapshots/26b840c1e723c2f2330ea298f6fd2c57c54eb888/Llama-3.1-8B-Lexi-Uncensored_V2_Q8.gguf",
#     context_window=16384,
#     max_new_tokens=1024,
#     generate_kwargs={
#                     # "do_sample": True, 
#                     #  "eos_token_id": tokenizer.eos_token_id,
#                      "top_k": 0,
#                      "top_p": 0,
#                      "temperature": 0,
#                      "repeat_penalty": 1.08
#                     },
#     messages_to_prompt=messages_to_prompt,
#     completion_to_prompt=completion_to_prompt,
#     # tokenizer_name="meta-llama/Llama-3.1-8B-Instruct",
#     # tokenizer_kwargs={"max_length": 2048},
#     model_kwargs={
#         "n_gpu_layers": -1,
#         # "grammar": grammar
#                  },
#     verbose = True
# )
# Settings.llm = llm

In [ ]:

llm = LlamaCPP(
    # You can pass in the URL to a GGML model to download it automatically
    # optionally, you can set the path to a pre-downloaded model instead of model_url
    # model_path="/hf_cache/models--NousResearch--Hermes-3-Llama-3.1-8B-GGUF/snapshots/307a5dfb59aa38d88b6cfd32f44b8ad7c1da9fb8/Hermes-3-Llama-3.1-8B.Q5_K_M.gguf",
    # model_url="https://huggingface.co/bartowski/DeepSeek-Coder-V2-Lite-Instruct-GGUF/blob/main/DeepSeek-Coder-V2-Lite-Instruct-Q6_K.gguf",
    model_path="/hf_cache/models--bartowski--Qwen2.5-Coder-14B-Instruct-GGUF/snapshots/5b379ec4bf71bafecb5f9081ad28b19939128988/Qwen2.5-Coder-14B-Instruct-Q6_K_L.gguf",
    temperature=0.1,
    max_new_tokens=4096,
    context_window=16384,
    generate_kwargs={
        "repeat_penalty": 1.1,
        "top_k": 0,
        "top_p": 0
    },
    model_kwargs={
        "n_gpu_layers": -1,
        # "grammar": grammar
                 },
    # messages_to_prompt=messages_to_prompt,
    # completion_to_prompt=completion_to_prompt,
    verbose=True,
)

Settings.llm = llm

In [ ]:
# llm.complete("Write a python class for manipulating wave audio nondestructively.")

In [ ]:
# llm = LlamaCPP(
#     # model_path = "/hf_cache/models--rombodawg--Rombos-LLM-V2.6-Qwen-14b-Q8_0-GGUF/snapshots/275ea57a28da4215ad43951ab858e4792d341591/rombos-llm-v2.6-qwen-14b-q8_0.gguf",
#     # model_path = "/hf_cache/models--arcee-ai--SuperNova-Medius-GGUF/snapshots/cd704022514371d420dd1690612660cd4b5072b2/SuperNova-Medius-Q6_K_L.gguf",
#     model_path = "/hf_cache/models--bartowski--Qwen2.5-14B_Uncensored_Instruct-GGUF/snapshots/2e7d5957ae9b9434ab58620f1073cae1fd0cf60a/Qwen2.5-14B_Uncensored_Instruct-Q6_K.gguf",
#     context_window=8192,
#     max_new_tokens=2048,
#     generate_kwargs={
#                     # "do_sample": True, 
#                     #  "eos_token_id": tokenizer.eos_token_id,
#                      "top_k": 0,
#                      "top_p": 0,
#                      "temperature": 0.1,
#                      "repeat_penalty": 1.08
#                     },
#     messages_to_prompt=messages_to_prompt,
#     completion_to_prompt=completion_to_prompt,
#     # tokenizer_name="meta-llama/Llama-3.1-8B-Instruct",
#     # tokenizer_kwargs={"max_length": 2048},
#     model_kwargs={
#         "n_gpu_layers": -1,
#         # "grammar": grammar
#                  },
#     verbose = True
# )
# Settings.llm = llm

In [ ]:
print(index.as_query_engine(response_mode="tree_summarize").query(f'Answer the following, given the provided context codebase for dealcloud push, a python library developed to map data between the dealcloud API and other services: Create a new Dealcloud model using the following json data example:{f}'))